# Part 4: ASL Video NMM Extraction

## Goal
Extract non-manual markers (NMMs) from ASL interpreter videos using MediaPipe Face Mesh. NMMs are the visual channel through which ASL interpreters convey epistemic stance: eyebrow raise signals uncertainty or questions, head tilt signals consideration, mouthing accompanies signs with English words. LLMs strip all of this out when they flatten hedging into confident text. This notebook quantifies what human interpreters do that LLMs cannot.

## How to use
1. Upload your .mp4 video files into a folder called `video_clips` in the Colab file browser (left sidebar). Or use the upload cell below.
2. Run each cell in order from top to bottom.
3. The notebook installs MediaPipe, loads videos, extracts facial landmarks per frame, computes brow height / head tilt / mouth aperture, generates time-series charts, and saves `youtube_nmm_analysis.json`.

## What it measures per frame
- **Brow height**: distance between eyebrow landmarks (70, 300) and eye landmarks (159, 386). Higher value = raised brows = potential epistemic marker.
- **Head tilt**: angle of the line between ear-region landmarks (234, 454). Deviation from 0 = tilted head.
- **Mouth aperture**: distance between upper lip (13) and lower lip (14). Higher = mouth open = mouthing.

## No API key needed. Runs entirely locally on Colab.

In [ ]:
# Step 1: Install dependencies and download face landmark model
!pip install mediapipe opencv-python-headless -q
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import json, os
from google.colab import files

!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

print('MediaPipe ready')

In [ ]:
# Step 2: Load videos
# Option A: If you already uploaded .mp4 files to video_clips/ in the file browser, just run this cell.
# Option B: If you have a zip file, uncomment the upload block below.

CLIP_DIR = 'video_clips'
os.makedirs(CLIP_DIR, exist_ok=True)

# --- Uncomment below to upload a zip ---
# print('Upload your zip of .mp4 files:')
# uploaded = files.upload()
# import zipfile
# for fname in uploaded:
#     if fname.endswith('.zip'):
#         with zipfile.ZipFile(fname) as z:
#             z.extractall(CLIP_DIR)
# --- End upload block ---

clips = [f for f in os.listdir(CLIP_DIR) if f.endswith('.mp4')]
print(f'Found {len(clips)} video clips:')
for c in sorted(clips):
    size = os.path.getsize(os.path.join(CLIP_DIR, c)) / 1e6
    print(f'  {c} ({size:.1f} MB)')

In [ ]:
# Step 3: NMM extraction function
# Uses MediaPipe Tasks API (works on current Colab without version conflicts)

def extract_nmm(video_path, sample_every=3):
    """Extract NMM features from video. Samples every N frames for speed."""
    base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        num_faces=1)
    detector = vision.FaceLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  ERROR: Cannot open {video_path}')
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    features = []
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % sample_every != 0:
            frame_idx += 1
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)
        if result.face_landmarks:
            lm = result.face_landmarks[0]
            features.append({
                'frame': frame_idx,
                'time_sec': round(frame_idx / fps, 2) if fps else 0,
                'brow_height': round((abs(lm[70].y - lm[159].y) + abs(lm[300].y - lm[386].y)) / 2, 5),
                'head_tilt_deg': round(np.degrees(np.arctan2(lm[454].y - lm[234].y, lm[454].x - lm[234].x)), 2),
                'mouth_open': round(abs(lm[14].y - lm[13].y), 5),
            })
        frame_idx += 1
        if frame_idx % 1000 == 0:
            print(f'    Frame {frame_idx}/{total}')
    cap.release()
    detector.close()
    return {'fps': fps, 'total_frames': total, 'sampled_frames': len(features), 'features': features}

print('Extraction function ready')

In [ ]:
# Step 4: Run extraction on all clips
clip_files = sorted([f for f in os.listdir(CLIP_DIR) if f.endswith('.mp4')])
print(f'Processing {len(clip_files)} clips\n')

all_nmm_data = {}
for i, vf in enumerate(clip_files):
    print(f'[{i+1}/{len(clip_files)}] {vf}')
    result = extract_nmm(os.path.join(CLIP_DIR, vf))
    if result and result['features']:
        all_nmm_data[vf] = result
        brows = [f['brow_height'] for f in result['features']]
        tilts = [f['head_tilt_deg'] for f in result['features']]
        mouths = [f['mouth_open'] for f in result['features']]
        threshold = np.mean(brows) + 1.5 * np.std(brows)
        raised_pct = len([b for b in brows if b > threshold]) / len(brows) * 100
        print(f'  Frames: {result["sampled_frames"]} / {result["total_frames"]}')
        print(f'  Brow: mean={np.mean(brows):.4f} std={np.std(brows):.4f} | Raised: {raised_pct:.1f}%')
        print(f'  Tilt: mean={np.mean(tilts):.1f} std={np.std(tilts):.1f}')
        print(f'  Mouth: mean={np.mean(mouths):.4f} std={np.std(mouths):.4f}')
    print()

print(f'Done. Processed {len(all_nmm_data)} clips.')

In [ ]:
# Step 5: Visualize NMM patterns per clip
import matplotlib.pyplot as plt

for vf, data in all_nmm_data.items():
    if not data['features']: continue
    times = [f['time_sec'] for f in data['features']]
    brows = [f['brow_height'] for f in data['features']]
    tilts = [f['head_tilt_deg'] for f in data['features']]
    mouths = [f['mouth_open'] for f in data['features']]

    fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
    fig.suptitle(vf, fontsize=10)

    threshold = np.mean(brows) + 1.5 * np.std(brows)
    axes[0].plot(times, brows, 'b-', linewidth=0.5)
    axes[0].axhline(np.mean(brows), color='r', linestyle='--', alpha=0.5, label='mean')
    axes[0].axhline(threshold, color='g', linestyle=':', alpha=0.5, label='1.5 std')
    axes[0].set_ylabel('Brow Height')
    axes[0].legend(fontsize=7)

    axes[1].plot(times, tilts, 'r-', linewidth=0.5)
    axes[1].axhline(0, color='k', linestyle='--', alpha=0.3)
    axes[1].set_ylabel('Head Tilt (deg)')

    axes[2].plot(times, mouths, 'g-', linewidth=0.5)
    axes[2].set_ylabel('Mouth Open')
    axes[2].set_xlabel('Time (sec)')

    plt.tight_layout()
    plt.savefig(f'nmm_{vf.replace(".mp4",".png")}', dpi=100)
    plt.show()

    raised_pct = len([b for b in brows if b > threshold]) / len(brows) * 100
    print(f'  Brow raised: {raised_pct:.1f}% | Tilt variability: {np.std(tilts):.2f} deg\n')

In [ ]:
# Step 6: Save results and summary table
output = {}
for vf, data in all_nmm_data.items():
    brows = [f['brow_height'] for f in data['features']]
    tilts = [f['head_tilt_deg'] for f in data['features']]
    mouths = [f['mouth_open'] for f in data['features']]
    threshold = np.mean(brows) + 1.5 * np.std(brows)
    output[vf] = {
        'total_frames': data['total_frames'],
        'sampled_frames': data['sampled_frames'],
        'fps': data['fps'],
        'duration_sec': round(data['total_frames'] / data['fps'], 1) if data['fps'] else 0,
        'brow_mean': round(float(np.mean(brows)), 5),
        'brow_std': round(float(np.std(brows)), 5),
        'brow_raise_pct': round(len([b for b in brows if b > threshold]) / len(brows) * 100, 1),
        'tilt_mean': round(float(np.mean(tilts)), 2),
        'tilt_std': round(float(np.std(tilts)), 2),
        'mouth_mean': round(float(np.mean(mouths)), 5),
        'mouth_std': round(float(np.std(mouths)), 5),
    }

with open('youtube_nmm_analysis.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f'{"Clip":<40} {"Duration":>8} {"Brow%":>8} {"Tilt":>8} {"Mouth":>8}')
print('-' * 76)
for clip, d in output.items():
    print(f'{clip[:39]:<40} {d["duration_sec"]:>7.1f}s {d["brow_raise_pct"]:>7.1f}% {d["tilt_std"]:>8.2f} {d["mouth_std"]:>8.5f}')

print(f'\nTotal clips: {len(output)}')
print(f'Brow raise range: {min(d["brow_raise_pct"] for d in output.values()):.1f}% - {max(d["brow_raise_pct"] for d in output.values()):.1f}%')
print(f'Tilt std range: {min(d["tilt_std"] for d in output.values()):.1f} - {max(d["tilt_std"] for d in output.values()):.1f} deg')

print('\nSaved: youtube_nmm_analysis.json')
files.download('youtube_nmm_analysis.json')

## How to interpret the results

**Brow raise %**: Percentage of frames where eyebrow height exceeds 1.5 standard deviations above the clip mean. In ASL, raised brows mark yes/no questions, conditional clauses, and epistemic uncertainty. 5-12% is typical for active signing.

**Tilt std (degrees)**: Standard deviation of head tilt angle across the clip. Higher values mean more head movement. Head tilt in ASL conveys consideration, doubt, and rhetorical questioning. 7-11 degrees is typical.

**Mouth std**: Standard deviation of mouth aperture. Higher values mean more active mouthing. ASL interpreters mouth English words alongside signs, especially for technical terms.

**The comparison**: LLMs in Parts 2-3 flatten hedging to 'high certainty' in flat text with zero visual markers. Human ASL interpreters actively use their face (brow raise, head tilt, mouthing) to convey the same epistemic information visually. That contrast is the multimodal finding of this project.